# 第四阶段：STL（标准模板库）

## 实验 5：`std::span` —— 借用连续元素

前面的 `std::vector` 和 `std::array` 都拥有连续元素，但它们的长度策略不同。若一个只读算法既要接受 C array、array，也要接受 vector，为每种 owner 分别编写重载会重复接口。

`std::span<T>` 把“首元素地址 + 元素数量”包装成非拥有视图。本实验关注：

- span 如何统一借用不同的连续 owner；
- `span<T>`、`span<const T>` 与 span 对象自身的 `const` 有何区别；
- 静态 extent 与动态 extent 分别能表达什么；
- owner 销毁或 vector 重分配后，span 为什么会失效；
- span 如何放在 C++ API 内部，并映射到 C ABI 与 Kotlin/Native。

In [ ]:
// 本步骤：引入 span、连续容器、算法和断言所需的标准库。
#include <algorithm>
#include <array>
#include <cassert>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <numeric>
#include <span>
#include <tuple>
#include <vector>

### 1. span 不拥有元素

动态长度 span 概念上只保存 pointer + length：

```text
owner: vector / array / C array
┌─────┬─────┬─────┬─────┐
│ T 0 │ T 1 │ T 2 │ T 3 │
└─────┴─────┴─────┴─────┘
   ▲                 ▲
   └── std::span<T> ─┘
       pointer + size
```

复制 span 只复制视图，不复制元素，也不会延长 owner 生命周期。span 析构时不会释放任何元素。因为它按值复制成本很低，函数参数通常直接写成 `std::span<const T> input`。

In [ ]:
// 本步骤：定义只读 span 接口，统一打印任意连续 int 序列。
void print_values(std::span<const int> values)
{
    // 通过 span 的 iterator 只读遍历 owner 的元素。
    for (int value : values)
    {
        std::cout << value << ' ';
    }

    std::cout << '\n';
}

In [ ]:
// 本步骤：从三种 owner 建立 span，验证一个接口可以统一读取它们。
{
    // 三个容器分别使用原生数组、固定长度 owner 和动态长度 owner。
    int c_array[]{1, 2, 3};
    std::array<int, 3> array{4, 5, 6};
    std::vector<int> vector{7, 8, 9};

    // 每次转换都只借用现有元素，不分配或复制底层序列。
    print_values(c_array);
    print_values(array);
    print_values(vector);

    // 原 owner 仍负责保存和释放各自的元素。
    assert(c_array[0] == 1);
    assert(array[0] == 4);
    assert(vector[0] == 7);
}

### 2. element type 决定能否修改 owner

`std::span<int>` 提供可写借用，可以修改底层 owner；`std::span<const int>` 提供只读借用。这里的 `const` 修饰元素类型，而不是创建元素快照。

可写 span 可以隐式转换成只读 span，反方向不成立。接口没有修改需求时应优先接收 `span<const T>`，把约束写进类型。

In [ ]:
// 本步骤：定义可写 span 接口，让修改直接作用于底层 owner。
void double_values(std::span<int> values)
{
    // value 是 owner 元素的引用，而不是循环中的临时副本。
    for (int &value : values)
    {
        value *= 2;
    }
}

In [ ]:
// 本步骤：通过可写 span 修改 array 和 vector，再从 owner 侧验证结果。
{
    std::array<int, 3> array{1, 2, 3};
    std::vector<int> vector{4, 5, 6};

    // 借用期间修改元素；span 不接管两个容器。
    double_values(array);
    double_values(vector);

    // 修改应直接反映到原 owner 中。
    assert((array == std::array<int, 3>{2, 4, 6}));
    assert(vector == std::vector<int>({8, 10, 12}));

    print_values(array);
    print_values(vector);
}

### 3. 区分“只读元素”和“const span 对象”

以下两个类型表达的约束不同：

```cpp
std::span<const int> view;       // 不能通过 view 修改 int
const std::span<int> view_object; // view 本身不能重新绑定，但 int 仍可修改
```

span 类似一个轻量 handle。handle 自身为 `const`，不代表它指向的对象也为 `const`。设计只读 API 时应把 `const` 放在 element type 上。

In [ ]:
// 本步骤：修改 const span handle 指向的元素，观察两层 const 的区别。
{
    std::array<int, 2> owner{10, 20};

    // view_object 不能重新赋值，但 element_type 仍是 int。
    const std::span<int> view_object = owner;
    view_object[0] = 99;

    // read_only 把相同 owner 暴露为只读元素。
    std::span<const int> read_only = owner;

    // 从 owner 和只读视图两侧验证前面的修改。
    assert(owner[0] == 99);
    assert(read_only[0] == 99);
}

### 4. 子视图仍然不复制元素

`first()`、`last()` 和 `subspan()` 只调整 view 的起始位置和长度。子视图继续借用同一个 owner，因此修改可写子视图会修改原序列。

span 的 `operator[]` 不执行边界检查。调用 `first(count)` 或 `subspan(offset, count)` 时也必须满足前置条件；越界不是可依赖异常恢复的普通错误。

In [ ]:
// 本步骤：创建前缀、中段和后缀视图，并验证它们共享同一 owner。
{
    std::array<int, 6> owner{10, 20, 30, 40, 50, 60};
    std::span<int> whole = owner;

    // 三个操作仅改变 pointer/length，不复制 array 元素。
    std::span<int, 2> first_two = whole.first<2>();
    std::span<int> middle = whole.subspan(2, 2);
    std::span<int, 2> last_two = whole.last<2>();

    // 通过中段视图写入，随后从 owner 观察相同位置。
    middle[0] = 300;

    assert(first_two[1] == 20);
    assert(owner[2] == 300);
    assert(last_two[0] == 50);

    // size_bytes 返回视图覆盖的字节数，不等于元素数量。
    assert(whole.size_bytes() == whole.size() * sizeof(int));
}

### 5. 动态 extent 与静态 extent

`std::span<T>` 等价于 `std::span<T, std::dynamic_extent>`，长度保存在运行期。写成 `std::span<T, 4>` 时，数字 4 进入类型系统，适合固定 header、矩阵行或固定通道数据。

静态 extent 只编码长度，不获得所有权。它仍然必须依赖一个存活足够久且至少包含对应元素的 owner。

In [ ]:
// 本步骤：为同一个固定 owner 建立静态和动态 extent 视图。
{
    std::array<std::uint8_t, 4> header{
        0x4b,
        0x4e,
        0x01,
        0x00};

    // 从 array 的编译期长度安全构造 static extent span。
    std::span<const std::uint8_t, 4> fixed = header;
    std::span<const std::uint8_t> dynamic = header;

    // extent 是类型级常量；size() 是当前视图的元素数量。
    static_assert(decltype(fixed)::extent == 4);
    static_assert(decltype(dynamic)::extent == std::dynamic_extent);
    assert(fixed.size() == dynamic.size());
    assert(fixed.data() == header.data());
}

### 6. owner 决定 span 的有效期

以下返回值在函数结束时立即悬空，因此只作为反例阅读：

```cpp
std::span<const int> bad_view()
{
    std::vector<int> local{1, 2, 3};
    return local; // local 析构并释放存储
}
```

即使 owner 仍存活，vector 重分配也会释放旧存储，使原 span 失效。array 没有扩容操作，因此元素地址更稳定，但 array 析构后 view 仍会悬空。

In [ ]:
// 本步骤：强制 vector 重分配，只比较地址数值并立即重建 span。
{
    std::vector<int> owner;
    owner.reserve(4);

    // 填满当前 capacity，保证下一次插入必须重新分配。
    while (owner.size() < owner.capacity())
    {
        owner.push_back(static_cast<int>(owner.size()));
    }

    std::span<const int> view = owner;
    const std::uintptr_t old_address =
        reinterpret_cast<std::uintptr_t>(view.data());

    // push_back 使旧 view 失效；从此绝不能读取或解引用旧 view。
    owner.push_back(99);
    const std::uintptr_t new_address =
        reinterpret_cast<std::uintptr_t>(owner.data());

    // 只比较扩容前保存的整数地址，不使用已失效的 pointer。
    assert(old_address != new_address);

    // 重新从有效 owner 建立 view，恢复合法借用关系。
    view = owner;
    assert(view.back() == 99);

    std::cout << "reallocated = "
              << std::boolalpha
              << (old_address != new_address)
              << '\n';
}

### 7. 用 span 表达算法真正需要的参数

求和算法不需要拥有容器，也不关心调用方选择 array、vector 还是 C array；它只要求一段连续、可读的 `int`。用 `span<const int>` 可以直接表达这个最小契约。

这比模板化整个函数更窄，也比裸 pointer 参数更完整，因为长度与地址绑定在同一个参数中。

In [ ]:
// 本步骤：定义只依赖连续只读序列的求和函数。
int sum_values(std::span<const int> values)
{
    // 把 span 的半开 iterator 区间交给标准算法。
    return std::accumulate(
        values.begin(),
        values.end(),
        0);
}

In [ ]:
// 本步骤：让不同 owner 复用同一个 span 算法接口。
{
    int c_array[]{1, 2, 3};
    std::array<int, 3> array{4, 5, 6};
    std::vector<int> vector{7, 8, 9};

    // 每次调用只在函数执行期间借用元素。
    const int c_sum = sum_values(c_array);
    const int array_sum = sum_values(array);
    const int vector_sum = sum_values(vector);

    // 验证三种 owner 都满足相同的连续只读契约。
    assert(c_sum == 6);
    assert(array_sum == 15);
    assert(vector_sum == 24);
}

### 8. span 属于 C++ 内部，C ABI 仍使用 pointer + length

`std::span` 是 C++ 标准库类型，不能直接出现在稳定 C ABI 中。边界函数应接收 C 可表达的参数，先验证 pointer + length，再在函数内部临时构造 span：

```text
Kotlin ByteArray / C buffer
          │ pointer + element count
          ▼
C ABI validates contract
          │
          ▼
temporary std::span<const uint8_t>
          │ read during call only
          ▼
return: span must not escape
```

In [ ]:
// 本步骤：定义 C++ span 算法和对应的无异常 C ABI 适配入口。
std::uint64_t checksum_bytes(
    std::span<const std::uint8_t> input) noexcept
{
    // 只读取 span 声明的元素范围，不保存 view。
    return std::accumulate(
        input.begin(),
        input.end(),
        std::uint64_t{0});
}

extern "C" bool sdk_checksum(
    const std::uint8_t *data,
    std::size_t size,
    std::uint64_t *result) noexcept
{
    // C 边界显式拒绝无效输出指针和非空长度对应的空输入指针。
    if (result == nullptr ||
        (data == nullptr && size != 0))
    {
        return false;
    }

    // 只在本次同步调用内包装借用，span 不得逃逸。
    const std::span<const std::uint8_t> input(data, size);
    *result = checksum_bytes(input);
    return true;
}

In [ ]:
// 本步骤：从 array 借出 pointer + length，验证一次完整的边界调用。
{
    const std::array<std::uint8_t, 4> payload{10, 20, 30, 40};
    std::uint64_t result = 0;

    // payload 覆盖完整调用，且调用期间没有修改 owner。
    const bool succeeded = sdk_checksum(
        payload.data(),
        payload.size(),
        &result);

    // 返回后 native 端不再保存借用，调用方可以安全继续使用 owner。
    assert(succeeded);
    assert(result == 100);

    std::cout << "checksum = " << result << '\n';
}

Kotlin/Native 调用时，`ByteArray` 的地址通常只能在 pinned/作用域借用期间传给 C。C++ 入口可以在同步调用内构造 span，但不能把 span 保存到成员、全局变量、其他线程或异步 callback。

如果数据需要在调用返回后继续使用，应复制到 native owner（例如 `std::vector<std::uint8_t>`），或设计明确的 create/destroy、retain/release 协议。span 本身不能把临时借用升级为长期所有权。

### 9. `as_bytes()` 观察对象表示，不等于序列化

`std::as_bytes()` 可以把对象 span 转换成只读字节视图，适合调用明确接收对象表示的底层 API。结果仍然借用原 owner，并可能包含平台相关的字节序、填充或类型表示。

因此 `as_bytes()` 不能自动产生跨平台文件格式或网络协议；稳定序列化必须明确字段宽度、字节序、布局和版本。

In [ ]:
// 本步骤：把整数 array 临时视为字节序列，只观察覆盖范围。
{
    std::array<std::uint32_t, 2> words{
        0x01020304u,
        0x05060708u};

    // as_bytes 不复制内容，只改变 view 的元素类型和长度单位。
    const std::span<const std::byte> bytes =
        std::as_bytes(std::span{words});

    // 字节视图覆盖两个 uint32_t 的完整对象表示。
    assert(bytes.size() == words.size() * sizeof(std::uint32_t));
    assert(bytes.data() ==
           reinterpret_cast<const std::byte *>(words.data()));

    std::cout << "byte count = " << bytes.size() << '\n';
}

### 10. 使用 span 前的检查清单

- 数据是否连续？`std::list` 等非连续容器不能转换成 span。
- 谁拥有元素？owner 是否覆盖 span 的整个使用周期？
- 借用期间是否会发生 vector 重分配、owner 移动或析构？
- callee 只读还是需要修改？优先选择 `span<const T>`。
- 长度是运行期状态还是固定协议？需要时使用 static extent，但边界仍应验证。
- 是否跨越同步调用、线程或异步任务？跨越时通常需要复制或所有权协议。
- `size()` 是元素数量，`size_bytes()` 才是覆盖的字节数。

### 本实验结论

`std::span<T, Extent>` 是连续元素的非拥有视图。它统一了 C array、`std::array` 和 `std::vector` 的借用接口，复制时只复制 pointer/length，不复制元素，也不参与释放。

element type 上的 `const` 决定是否能通过 view 修改 owner；静态 extent 把长度放进类型，但两者都不会延长 owner 生命周期。owner 析构、vector 重分配或其他失效操作都会让相关 span 悬空。

span 适合 C++ 内部同步处理 buffer。跨 C ABI 时仍使用经过验证的 pointer + length；跨 Kotlin/Native pinned 作用域、线程或异步边界时，应复制数据或建立明确的长期所有权协议。